## Tmid Analysis: Investigating how Tmid is calculated between SWSPy and Baillard methods
We will get the output Tmid for SWSPy, Baillard on the same events and compare

### 1. Dependencies

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import obspy
from obspy.core.utcdatetime import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.event import read_events
import os
import sys
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add local swspy directory to path (before other imports)
swspy_local_path = os.path.abspath('../swspy')
if swspy_local_path not in sys.path:
    sys.path.insert(0, swspy_local_path)

# Import swspy from local directory
import swspy

# Add scripts directory to path for custom modules
sys.path.append('.')
from get_all_traces import get_station_traces_batch
from splitting_functions import *
from teanby_clustering import *

# Set up plotting
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully")
print(f"ObsPy version: {obspy.__version__}")
print(f"SWSPy available: {'Yes' if 'swspy' in sys.modules else 'No'}")
print(f"SWSPy location: {swspy.__file__}")

In [ ]:
# Load Baillard nonlinloc catalog
catalog = pd.read_csv('AXIAL.PHASE.FINAL_3D_V2.csv')

#Load station information from Christian's data
stations_file = '../data/stations_axial.llz'
#Read llz file - reads like a text file with space delimiter
stations_df = pd.read_csv(stations_file, delim_whitespace=True, header=None, names=['Longitude (°W)', 'Latitude (°N)', 'Elevation (m)', 'Station ID'])
# Convert elevation column to m from km
stations_df['Elevation (m)'] = stations_df['Elevation (m)']*1000

print(f"Stations in catalog: {catalog['station'].value_counts()}")

In [ ]:
# Filter catalog for AXAS2 station only

axas2_catalog = catalog[catalog['station'] == 'AXAS2'].copy()


# Reset index to ensure clean indexing
axas2_catalog = axas2_catalog.reset_index(drop=True)

# Pick all events from April and May, 2015
axas2_catalog['datetime'] = pd.to_datetime(axas2_catalog['datetime'])
axas2_catalog = axas2_catalog[(axas2_catalog['datetime'] >= '2015-04-20') & (axas2_catalog['datetime'] < '2015-04-28')].copy()


# Select first 100 events for testing
test_catalog_100 = axas2_catalog.head(10).copy()
print(f"Total AXAS2 events in catalog: {len(axas2_catalog)}")

display(test_catalog_100)

In [ ]:
# Now convert to pandas Timestamp with UTC timezone
test_catalog_100['p_time'] = pd.to_datetime(test_catalog_100['p_time'], utc=True, format='ISO8601')
test_catalog_100['s_time'] = pd.to_datetime(test_catalog_100['s_time'], utc=True, format='ISO8601')
test_catalog_100['datetime'] = pd.to_datetime(test_catalog_100['datetime'], utc=True, format='ISO8601')

# Convert to UTCDateTime
test_catalog_100['p_time'] = test_catalog_100['p_time'].apply(lambda x: UTCDateTime(x))
test_catalog_100['s_time'] = test_catalog_100['s_time'].apply(lambda x: UTCDateTime(x))
test_catalog_100['datetime'] = test_catalog_100['datetime'].apply(lambda x: UTCDateTime(x))

In [ ]:
catalog = test_catalog_100.copy()

In [ ]:
# Create extended time windows for proper waveform analysis
print("Creating extended time windows for waveform retrieval...")

# Apply extended windowing
extended_catalog = create_extended_catalog(catalog, pre_p_time=1.0, post_s_time=2.0)

print(f"Extended catalog created with {len(extended_catalog)} events")
print(f"Time windows: {extended_catalog['total_duration'].iloc[0]} seconds total")
print(f"Pre-event: {extended_catalog['pre_p_sec'].iloc[0]}s, Post-event: {extended_catalog['post_s_sec'].iloc[0]}s")
# Display sample of extended timing
print("\nSample timing windows:")
sample_cols = ['id', 'datetime', 'starttime', 'endtime', 'total_duration']
display(extended_catalog[sample_cols].head())

In [ ]:
display(extended_catalog)

In [ ]:
test_catalog = extended_catalog
# Replace catalog id with index
test_catalog['id'] = test_catalog.index

test_catalog['mag'] = 0.0

In [ ]:
waveforms = get_station_traces_batch(test_catalog, 'axial_nonlinloc_april_100', 'starttime', 'endtime', 'station', batch_size=100)

In [ ]:
# Associate waveforms with events in the catalog
print("Organizing waveforms by events...")
waveform_dict = organize_stream_by_events(waveforms, test_catalog)

In [ ]:
# Organize waveforms by event ID
print("Organizing waveforms by event ID...")
organized_waveforms = organize_waveform_data(waveform_dict, test_catalog)

In [ ]:
# Format s_arrival_time and p_arrival_time as difference between arrival times and origin time
print("Formatting s_arrival_time and p_arrival_time as differences from origin time...")
for eid in organized_waveforms.keys():
    organized_waveforms[eid]['s_arrival_time'] = (UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 's_time'].values[0]) - 
                                                  UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'datetime'].values[0]))
    organized_waveforms[eid]['p_arrival_time'] = (UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'p_time'].values[0]) - 
                                                  UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'datetime'].values[0]))

In [ ]:
# For all traces in organized_waveforms, taper and filter in-place
print("Tapering and filtering all traces in organized_waveforms...")
events_to_remove = []
try:
    for eid in organized_waveforms.keys():
        for tr in organized_waveforms[eid]['traces']:
            tr.detrend("linear") # to avoid weird start and end amplitudes
            tr.taper(max_percentage=0.05, type='hann')
            tr.filter('bandpass', freqmin=5.0, freqmax=40.0)
except Exception as e:
    print(f"Error during waveform processing: {e}")
    if type(organized_waveforms[eid]['traces']) == type(None):
        events_to_remove.append(eid)
    print(f"Events with issues: {events_to_remove}")

print("Waveform retrieval and organization complete.")

In [ ]:
# Remove duplicate traces from organized_waveforms
print("Checking for and removing duplicate traces in organized_waveforms...")

for eid in organized_waveforms.keys():
    # Get the stream for this event
    st = organized_waveforms[eid]['traces']
    
    # Check if there are duplicates
    try:
        if len(st) > 3:
            print(f"Event {eid}: Found {len(st)} traces (expected 3)")
            
            # Create a new stream with unique traces based on channel code
            unique_traces = {}
            for tr in st:
                channel = tr.stats.channel
                # Keep the first occurrence of each channel
                if channel not in unique_traces:
                    unique_traces[channel] = tr
            
            # Replace the stream with deduplicated traces
            organized_waveforms[eid]['traces'] = obspy.Stream(traces=list(unique_traces.values()))
            print(f"  Reduced to {len(organized_waveforms[eid]['traces'])} unique traces")
    except Exception as e:
        print(f"Error processing event {eid}: {e}")
        organized_waveforms[eid]['traces'] = st[:3]  # Fallback to first 3 traces if error occurs

# Verify the results
print("\nVerification of trace counts after deduplication:")
trace_counts = {}
for eid in organized_waveforms.keys():
    count = len(organized_waveforms[eid]['traces'])
    trace_counts[count] = trace_counts.get(count, 0) + 1

print(f"Events with 3 traces: {trace_counts.get(3, 0)}")
if any(k != 3 for k in trace_counts.keys()):
    print("Events with unexpected trace counts:")
    for count, num_events in trace_counts.items():
        if count != 3:
            print(f"  {num_events} events with {count} traces")
else:
    print("All events have exactly 3 traces (E, N, Z)")

In [ ]:
# Define quality control thresholds
QC_THRESHOLDS = {
    'min_snr': 2.0,           # Minimum S-wave signal-to-noise ratio
    'min_rectilinearity': 0.7, # Minimum P-wave rectilinearity
    'max_incidence': 30.0,     # Maximum incidence angle (degrees)
    'min_magnitude': 0.0,      # Minimum event magnitude
}

print("Quality control functions loaded successfully")
print(f"QC Thresholds: {QC_THRESHOLDS}")

In [ ]:
# Check that all traces for same event have same length, and remove events that do not meet this criterion
print("Checking that all traces for the same event have the same length...")
events_to_remove = []
for eid in organized_waveforms.keys():
    trace_lengths = [tr.stats.npts for tr in organized_waveforms[eid]['traces']]
    if len(set(trace_lengths)) != 1:
        print(f"Event {eid} has traces of different lengths: {trace_lengths}, marking for removal")
        events_to_remove.append(eid)

In [ ]:
# Remove traces for events that do not have consistent trace lengths
for eid in events_to_remove:
    print(f"Removing event {eid} due to inconsistent trace lengths")
    del organized_waveforms[eid]

In [ ]:
def get_dominant_period_baillard(data, fs, method='welch', nfft=256, num_wind=2, flag_plot=False):
    """
    Function made to compute the dominant period of a 1D np.array using either
    a classical FFT or a multitaper methods
    
    Output
    ------
        dom_period: float: dominant period in samples (can be float)
        dom_freq: float: dominant frequency
    
    """
    
    ### Compute
    
    if method=='welch':
        freq,spec=signal.welch(data, fs=fs,nperseg=len(data)/num_wind,scaling='density',nfft=nfft)
    #elif method=='multitaper' or method=='mtspec':
    #    if not MULTITAPER_AVAILABLE:
    #        raise ImportError("multitaper package is not installed. Install with: pip install multitaper")
        # multitaper package usage: MTSpec(data, nw, k, dt)
        # nw = time-bandwidth product (typically 2-4), k = number of tapers (typically 2*nw - 1)
    #    nw = num_wind if num_wind > 1 else 2.5
    #    k = int(2 * nw - 1)
    #    dt = 1.0 / fs
    #    mtspec_obj = MTSpec(data, nw=nw, k=k, dt=dt, nfft=nfft)
    #    spec = mtspec_obj.rspec()  # Return spectrum
    #    freq = mtspec_obj.freq     # Return frequencies
    #elif method=='classic':
    #    ([freq,spec],_)=spectrum(data,1/fs,nfft=nfft,flag_plot=False)
        
    ### Freq to period
      
    with np.errstate(divide='ignore'):
        period=1/freq*fs
    
    ### Get Dominant period

    dom_period=period[np.argmax(spec)]  
    dom_freq=1/dom_period*fs
    
    ### Plot if asked
    
    if flag_plot:
        x=np.arange(0,len(data))
        alpha=2*np.pi/dom_period
        sin=np.max(data)*np.sin(alpha*x)
            
        fig,ax=plt.subplots(3,1)
        
        ax[0].plot(x,data,color='k',ls='-',lw=1)
        ax[0].plot(x,sin,color='r',ls='--',lw=1)
        
        ax[1].plot(period,spec,color='k',ls='-',lw=1)
        ax[1].axvline(dom_period,color='r',ls='--')
        ax[1].text(0.5, 0.9,'Dom T=%.1f samples'%dom_period, horizontalalignment='center',
                  verticalalignment='center', transform=ax[1].transAxes)
        
        ax[2].plot(freq,spec,color='k',ls='-',lw=1)
        ax[2].axvline(dom_freq,color='r',ls='--')
        ax[2].text(0.5, 0.9,'Dom F=%.1f Hz'%dom_freq, horizontalalignment='center',
                  verticalalignment='center', transform=ax[2].transAxes)
        
    return (dom_period,dom_freq)

In [ ]:
def compute_snr_for_event_baillard(event_stream, event_row, 
                                   N_left=80, N_right=40, mode='mean',
                                   apply_filter=False, check_p_contamination=True,
                                   flag_adapt_window=True, flag_adapt_maxlag=True):
    
    s_window=[0.02,0.3]
    fs_window=[0.1,0.3]
    s_snr_window=[0.4,0.2]

    st_x=event_stream.select(channel='??E')
    st_y=event_stream.select(channel='??N')
    st_xy=st_x+st_y
    xy_array=stream2data(st_xy)

    p_time = UTCDateTime(event_row['datetime']) + float(event_row['p_arrival_time'])
    s_time = UTCDateTime(event_row['datetime']) + float(event_row['s_arrival_time'])
    fs_window_time=[s_time-fs_window[0],s_time+fs_window[1]] 
    s_window_time=[s_time-s_window[0],s_time+s_window[1]] 
    trace_start_time=st_xy[0].stats.starttime
    sampling_rate=st_xy[0].stats.sampling_rate
    sampling_rate=event_stream[0].stats.sampling_rate

    s_samples=int(round((s_time-trace_start_time)*sampling_rate)) # in samples
    p_samples=int(round((p_time-trace_start_time)*sampling_rate)) # in samples
    fs_w1=int(round((fs_window_time[0]-trace_start_time)*sampling_rate))
    fs_w2=int(round((fs_window_time[1]-trace_start_time)*sampling_rate))
    mid_samples=int(round(p_samples+(s_samples-p_samples)/2))
    sw1=int(round((s_window_time[0]-trace_start_time)*sampling_rate))
    sw2=int(round((s_window_time[1]-trace_start_time)*sampling_rate))

    if fs_w1<mid_samples:
        fs_w1=mid_samples
                
        ### Cut the data between fs_w1 and fs_w2
        
        xy_array_dom=xy_array[fs_w1:fs_w2,:]
        
        ### Get dominant period on X and Y (given in samples)
        
        (dom_period_x,dom_freq_x)=get_dominant_period_baillard(xy_array_dom[:,0],sampling_rate,flag_plot=False)
        (dom_period_y,dom_freq_y)=get_dominant_period_baillard(xy_array_dom[:,1],sampling_rate,flag_plot=False)
    
        dom_period=np.mean([dom_period_x,dom_period_y]) # Take the mean dominant period
        
        ### Adapt window size to perform splitting and max lag allowed
        
        if flag_adapt_window:
            sw2=int(round(sw1+2*dom_period)) # Check Wuestfeld et al., 2010
        else:
            sw2=int(round((s_window_time[1]-trace_start_time)*sampling_rate))
            
        if flag_adapt_maxlag:
            max_lag=int(round(dom_period))

    else:
        dom_period = 0

    s_snr_window_time=[s_time-s_snr_window[0],s_time+s_snr_window[1]] 
    s_snr_w1=int(round((s_snr_window_time[0]-trace_start_time)*sampling_rate))
    s_snr_w2=int(round((s_snr_window_time[1]-trace_start_time)*sampling_rate))

    ### Make sure left side of the window is bigger than P+(S-P)/2
    if s_snr_w1<mid_samples:
        s_snr_w1=mid_samples
        
    if s_snr_w2>sw2:
        s_snr_w2=sw2

    ### Compute

    snr_x=SNR_pick(xy_array[:,0],s_samples,
                        s_samples-s_snr_w1,s_snr_w2-s_samples,mode='mean')
    snr_y=SNR_pick(xy_array[:,1],s_samples,
                        s_samples-s_snr_w1,s_snr_w2-s_samples,mode='mean')

    ### Feed obs

    s_snr=np.mean((snr_x,snr_y))


    return s_snr, dom_period

In [ ]:
def calculate_snr_for_organized_waveforms(organized_waveforms):
    """
    Calculate SNR for all events in organized_waveforms and add to the dataset.
    
    All required metadata (S/P arrival times, datetime) is already in organized_waveforms,
    so no external catalog lookup is needed.
    
    Parameters:
    -----------
    organized_waveforms : dict
        Dictionary with event IDs as keys, containing event data, traces, and metadata
        
    Returns:
    --------
    dict
        Updated organized_waveforms with SNR values added to each event
    """
    
    print(f"Calculating SNR for {len(organized_waveforms)} events in organized_waveforms...")
    
    success_count = 0
    
    for event_id, event_data in organized_waveforms.items():
        print(f"\n{'='*60}")
        print(f"Processing event {event_id}...")
        print(f"{'='*60}")
        
        # Get traces for this event
        event_traces = event_data.get('traces', [])
        if not event_traces:
            print(f"  No traces found for event {event_id}")
            event_data['snr_e'] = np.nan
            event_data['snr_n'] = np.nan
            event_data['snr_horizontal'] = np.nan
            continue
        
        print(f"  Found {len(event_traces)} traces for this event")
        
        # Calculate SNR for horizontal components using event_data directly
        print("\n  Calculating SNR for E component:")
        snr_horizontal, dom_period = compute_snr_for_event_baillard(event_traces.copy(), event_data)
  
        event_data['snr_horizontal'] = snr_horizontal
        event_data['dominant_period_baillard'] = dom_period
        
        # Print summary
        print(f"\n  Final SNR Results:")

        if not np.isnan(snr_horizontal):
            print(f"    Horizontal average: {snr_horizontal:.2f}")
        else:
            print("    Horizontal average: N/A")

        if not np.isnan(snr_horizontal):
            success_count += 1
    
    print(f"\n{'='*60}")
    print("SNR Calculation Complete")
    print(f"{'='*60}")
    print(f"Events with valid SNR: {success_count}/{len(organized_waveforms)}")
    
    # Calculate statistics
    snr_values = [data.get('snr_horizontal', np.nan) for data in organized_waveforms.values()]
    valid_snr = [v for v in snr_values if not np.isnan(v)]
    
    if valid_snr:
        print(f"SNR range: {min(valid_snr):.2f} to {max(valid_snr):.2f}")
        print(f"Mean SNR: {np.mean(valid_snr):.2f}")
        print(f"Median SNR: {np.median(valid_snr):.2f}")
    
    return organized_waveforms

In [ ]:
# Redefine calculate_snr_for_organized_waveforms to skip events that return empty spec sequence, then remove them from organized_waveforms

def calculate_snr_for_organized_waveforms(organized_waveforms):
    """
    Calculate SNR for all events in organized_waveforms and add to the dataset.
    
    All required metadata (S/P arrival times, datetime) is already in organized_waveforms,
    so no external catalog lookup is needed.
    
    Parameters:
    -----------
    organized_waveforms : dict
        Dictionary with event IDs as keys, containing event data, traces, and metadata
        
    Returns:
    --------
    dict
        Updated organized_waveforms with SNR values added to each event
    """
    
    print(f"Calculating SNR for {len(organized_waveforms)} events in organized_waveforms...")
    
    success_count = 0
    
    for event_id, event_data in organized_waveforms.items():
        print(f"\n{'='*60}")
        print(f"Processing event {event_id}...")
        print(f"{'='*60}")
        
        # Get traces for this event
        event_traces = event_data.get('traces', [])
        if not event_traces:
            print(f"  No traces found for event {event_id}")
            event_data['snr_e'] = np.nan
            event_data['snr_n'] = np.nan
            event_data['snr_horizontal'] = np.nan
            continue
        
        print(f"  Found {len(event_traces)} traces for this event")
        
        # Calculate SNR for horizontal components using event_data directly

        try:
            print("\n  Calculating SNR for E component:")
            snr_horizontal, dom_period = compute_snr_for_event_baillard(event_traces.copy(), event_data)
    
        except Exception as e:
            print(f"  Error computing SNR for event {event_id}: {e}")
            snr_horizontal = np.nan

        event_data['snr_horizontal'] = snr_horizontal
        event_data['dominant_period_baillard'] = dom_period
        
        # Print summary
        print(f"\n  Final SNR Results:")

        if not np.isnan(snr_horizontal):
            print(f"    Horizontal average: {snr_horizontal:.2f}")
        else:
            print("    Horizontal average: N/A")

        if not np.isnan(snr_horizontal):
            success_count += 1
    
    print(f"\n{'='*60}")
    print("SNR Calculation Complete")
    print(f"{'='*60}")
    print(f"Events with valid SNR: {success_count}/{len(organized_waveforms)}")
    
    # Calculate statistics
    snr_values = [data.get('snr_horizontal', np.nan) for data in organized_waveforms.values()]
    valid_snr = [v for v in snr_values if not np.isnan(v)]
    
    if valid_snr:
        print(f"SNR range: {min(valid_snr):.2f} to {max(valid_snr):.2f}")
        print(f"Mean SNR: {np.mean(valid_snr):.2f}")
        print(f"Median SNR: {np.median(valid_snr):.2f}")
    
    else:
        # Remove events that have NaN SNR values from organized_waveforms
        print("No valid SNR values found, removing events with NaN SNR from organized_waveforms...")
        events_to_remove = [eid for eid, data in organized_waveforms.items() if np.isnan(data.get('snr_horizontal', np.nan))]
        for eid in events_to_remove:
            del organized_waveforms[eid]
        print(f"Removed {len(events_to_remove)} events with NaN SNR values")
        print(f"Remaining events after SNR QC: {len(organized_waveforms)}")
    return organized_waveforms

In [ ]:
# Calculate quality control metrics for organized waveforms
print("Calculating quality control metrics for organized waveforms...")

# 1. Calculate S-wave SNR
organized_waveforms = calculate_snr_for_organized_waveforms(organized_waveforms)

# 2. Calculate geographic back-azimuth, for coordinate rotation later
organized_waveforms = calculate_back_azimuth_for_organized_waveforms(organized_waveforms, stations_df)

# 3. Calculate incidence angle
organized_waveforms = calculate_incidence_angle_eigenvalue_jurkevics_for_organized_waveforms(organized_waveforms, p_arrival_variable='p_arrival_time', analysis_window=0.12)

#4. Calculate P-wave rectilinearity
organized_waveforms = calculate_rectilinearity_jurkevics_for_organized_waveforms(organized_waveforms, p_arrival_variable='p_arrival_time', analysis_window=0.12)

In [ ]:
# Define passing_waveforms as those that meet all QC thresholds
passing_waveforms = apply_quality_control(organized_waveforms, QC_THRESHOLDS)

In [ ]:
# Plot a histogram of organized_waveforms['dominant_period_baillard']

dominant_periods = [data.get('dominant_period_baillard', np.nan) for data in organized_waveforms.values()]

In [ ]:
dominant_freqs_x = [data.get('dominant_frequency_x_baillard', np.nan) for data in organized_waveforms.values()]
dominant_freqs_y = [data.get('dominant_frequency_y_baillard', np.nan) for data in organized_waveforms.values()]

In [ ]:
# Filter dominant periods > 0


In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(dominant_periods, bins=20, color='skyblue', edgecolor='black')
plt.title('Histogram of Dominant Periods (Baillard Method)')
plt.xlabel('Dominant Period (Samples)')
plt.ylabel('Number of Events')
plt.grid(True)
plt.show()

In [ ]:
def perform_splitting_on_organized_waveforms(organized_waveforms, use_dynamic_params=False, mode='swspy', plot_results=False):
    """
    Perform shear-wave splitting analysis on all events in organized_waveforms.
    
    This function assumes organized_waveforms has been filtered by apply_quality_control()
    and contains only events that pass QC thresholds. It extracts horizontal components
    and performs splitting analysis using the back_azimuth for coordinate rotation.
    
    Parameters:
    -----------
    organized_waveforms : dict
        Dictionary with event IDs as keys, containing QC-filtered traces and metadata
        Must have: traces, back_azimuth, station, and all other event metadata

    mode : str
        Splitting analysis method to use: 'swspy' or 'baillard' (default: 'swspy')
        
    Returns:
    --------
    dict
        Dictionary with event IDs as keys, each containing:
        {
            event_id: {
                'result': result_dict,  # Splitting parameters and metadata
                'splitting_obj': splitting_obj  # SWSPy splitting object
            },
            ...
        }
    """
    
    print(f"\n{'='*60}")
    print("Performing Shear-Wave Splitting Analysis")
    print(f"{'='*60}")
    print(f"Processing {len(organized_waveforms)} QC-filtered events...")
    
    event_results = {}
    
    # Track statistics
    stats = {
        'total_events': len(organized_waveforms),
        'missing_components': 0,
        'missing_back_azimuth': 0,
        'splitting_errors': 0,
        'successful_splits': 0
    }
    
    for event_id, event_data in organized_waveforms.items():
        print(f"\n{'─'*60}")
        print(f"Event {event_id}")
        print(f"{'─'*60}")
        
        # Get traces
        event_traces = event_data.get('traces', [])
        if not event_traces:
            print(f"  ✗ No traces found")
            stats['missing_components'] += 1
            continue
        
        # Convert to stream if needed
        if isinstance(event_traces, list):
            event_stream = obspy.Stream(event_traces)
        else:
            event_stream = event_traces
        
        # Find N and E components
        trace_n = None
        trace_e = None
        trace_z = None
        
        for tr in event_stream:
            component = tr.stats.channel[-1].upper()
            if component in ['N', '1']:
                trace_n = tr
            elif component in ['E', '2']:
                trace_e = tr
            elif component == 'Z':
                trace_z = tr
        
        # Check if we have horizontal components
        if trace_n is None or trace_e is None:
            print(f"  ✗ Missing horizontal components (N: {trace_n is not None}, E: {trace_e is not None})")
            stats['missing_components'] += 1
            continue
        
        print(f"  ✓ Found horizontal components")
        
        # Check for back-azimuth
        back_azimuth = event_data.get('back_azimuth')
        if back_azimuth is None or np.isnan(back_azimuth):
            print(f"  ✗ Missing back-azimuth")
            stats['missing_back_azimuth'] += 1
            continue
        
        print(f"  ✓ Back-azimuth: {back_azimuth:.2f}°")
        
        # Get station and other metadata
        station_name = event_data.get('station', 'UNKNOWN')
        magnitude = event_data.get('magnitude', np.nan)
        snr_horizontal = event_data.get('snr_horizontal', np.nan)
        rectilinearity = event_data.get('rectilinearity_jurkevics', np.nan)
        incidence = event_data.get('incidence_eigenvalue_jurkevics', np.nan)
        
        print(f"  Station: {station_name}")
        print(f"  Magnitude: {magnitude:.1f}")
        print(f"  SNR: {snr_horizontal:.2f}")
        print(f"  Rectilinearity: {rectilinearity:.3f}")
        print(f"  Incidence: {incidence:.1f}°")
        
        # Perform splitting analysis using Baillard method
        print(f"\n  → Running Baillard splitting analysis...")
        
        try:
           
            if mode=='baillard':
                # Call Baillard splitting function
                splitting_result = perform_splitting_analysis_baillard(
                    event_data,
                    s_window=[0.02,0.3],
                    min_lag=0,
                    max_lag=60,
                    Nlags=60,
                    Nangles=90,
                    flag_adapt_window=True,
                    flag_adapt_maxlag=True,
                    min_thres=0.5,
                    min_numbers=2,
                    plot_results=plot_results
                )
                
                # Baillard returns dict only (no splitting_obj like SWSPy)
                splitting_obj = None

            # Hemmett-adapted SWSPy-like implementation that draws on MFAST
            elif mode=='swspy':
                # Call SWSPy-like splitting function
                splitting_result, splitting_obj = perform_splitting_analysis(
                    event_data, use_dynamic_params=use_dynamic_params, plot_results=plot_results
                )

            elif mode=='teanby_baillard':
                # Call Teanby clustering with Baillard method
                splitting_result = tbc.teanby_clustering_analysis(
                    event_data,
                    T_beg_1=0.02, T_end_0=0.3,
                    dT_beg=0.005, dT_end=0.005,
                    N_beg=10, N_end=10,
                    min_lag=0, max_lag=60,
                    Nlags=60, Nangles=90,
                    flag_adapt_window=True,
                    flag_adapt_maxlag=True,
                    min_thres=0.5,
                    min_numbers=2,
                    plot_results=plot_results,
                    output_dir=None,
                    N_c_min=5
                )
                splitting_obj = None

            if splitting_result.get('success', False):
                # Add additional metadata to result
                splitting_result['event_id'] = event_id
                splitting_result['back_azimuth'] = back_azimuth
                splitting_result['snr_horizontal'] = snr_horizontal
                splitting_result['rectilinearity_jurkevics'] = rectilinearity
                splitting_result['incidence_eigenvalue_jurkevics'] = incidence
                splitting_result['event_lat'] = event_data.get('latitude')
                splitting_result['event_lon'] = event_data.get('longitude')
                splitting_result['event_depth'] = event_data.get('depth')
                splitting_result['event_datetime'] = event_data.get('datetime')
                
                # Store both result dict and splitting object for later use
                event_results[event_id] = {
                    'result': splitting_result,
                    'splitting_obj': splitting_obj
                }
                stats['successful_splits'] += 1
                
                print(f"  ✓ SUCCESS!")
                if not np.isnan(splitting_result['phi']):
                    print(f"    Fast axis (φ): {splitting_result['phi']:.1f}°")
                else:
                    print(f"    Fast axis (φ): N/A")
                    
                if not np.isnan(splitting_result['dt']):
                    print(f"    Delay time (δt): {splitting_result['dt']:.3f}s")
                else:
                    print(f"    Delay time (δt): N/A")
                    
                if not np.isnan(splitting_result.get('phi_error', np.nan)):
                    print(f"    φ error: ±{splitting_result['phi_error']:.1f}°")
                if not np.isnan(splitting_result.get('dt_error', np.nan)):
                    print(f"    δt error: ±{splitting_result['dt_error']:.3f}s")
                if 'dominant_period' in splitting_result:
                    print(f"    Dominant period: {splitting_result['dominant_period']:.3f}s")
            else:
                error_msg = splitting_result.get('error', 'Unknown error')
                print(f"  ✗ FAILED: {error_msg}")
                if 'traceback' in splitting_result:
                    print(f"  Traceback:\n{splitting_result['traceback']}")
                stats['splitting_errors'] += 1
                
        except Exception as e:
            import traceback
            print(f"  ✗ ERROR during splitting: {e}")
            print(f"  Traceback:\n{traceback.format_exc()}")
            stats['splitting_errors'] += 1
            continue
    
    # Print summary
    print(f"\n{'='*60}")
    print("Splitting Analysis Summary")
    print(f"{'='*60}")
    print(f"Total events processed: {stats['total_events']}")
    print(f"Successful splits: {stats['successful_splits']} ({100*stats['successful_splits']/stats['total_events']:.1f}%)")
    print(f"\nFailure breakdown:")
    print(f"  Missing components: {stats['missing_components']}")
    print(f"  Missing back-azimuth: {stats['missing_back_azimuth']}")
    print(f"  Splitting errors: {stats['splitting_errors']}")
    
    # Calculate splitting parameter statistics if we have results
    if event_results:

        if mode!='swspy':
            # Extract phi and dt values from nested structure
            phi_values = [r['result']['phi_rad'] for r in event_results.values() 
                        if not np.isnan(r['result']['phi_rad'])]
            dt_values = [r['result']['dt'] for r in event_results.values() 
                        if not np.isnan(r['result']['dt'])]
        
        else:
            # Extract phi and dt values from nested structure
            phi_values = [r['result']['phi'] for r in event_results.values() 
                        if not np.isnan(r['result']['phi'])]
            dt_values = [r['result']['dt'] for r in event_results.values() 
                        if not np.isnan(r['result']['dt'])]
        
        if phi_values and dt_values:
            print(f"\n{'─'*60}")
            print("Splitting Parameter Statistics")
            print(f"{'─'*60}")
            print(f"Fast axis direction (φ):")
            print(f"  Mean: {np.mean(phi_values):.1f}° ± {np.std(phi_values):.1f}°")
            print(f"  Range: {np.min(phi_values):.1f}° to {np.max(phi_values):.1f}°")
            print(f"  Median: {np.median(phi_values):.1f}°")
            
            print(f"\nDelay time (δt):")
            print(f"  Mean: {np.mean(dt_values):.3f} ± {np.std(dt_values):.3f}s")
            print(f"  Range: {np.min(dt_values):.3f}s to {np.max(dt_values):.3f}s")
            print(f"  Median: {np.median(dt_values):.3f}s")
            
            # Add to stats
            stats['phi_mean'] = float(np.mean(phi_values))
            stats['phi_std'] = float(np.std(phi_values))
            stats['dt_mean'] = float(np.mean(dt_values))
            stats['dt_std'] = float(np.std(dt_values))
        else:
            print(f"\n  Note: No valid splitting parameters for statistics")
    
    # Return the event_results dictionary directly
    # Each entry contains both 'result' and 'splitting_obj'
    return event_results

In [ ]:
# For event in organized_waveforms, initialize the object with create_splitting_object such that T_mid will be created
# Then extract the values of T_mid for the splitting object, define as dominant_period_swspy in organized_waveforms, and compare to dominant_period_baillard


In [ ]:
#import swspy.splitting.split_hemmett as shs

In [ ]:
def create_splitting_analysis(event_data, use_dynamic_params=True):
    """
    Create a SWSPy splitting object from organized_waveforms event data.
    
    All required data (traces, station, back_azimuth, incidence, s_arrival_time) is 
    contained in event_data from organized_waveforms.
    
    Parameters:
    -----------
    event_data : dict
        Event data from organized_waveforms containing:
        - traces: ObsPy stream with waveform data
        - station: Station name/ID
        - back_azimuth: Back-azimuth from event to station (degrees)
        - incidence_eigenvalue_jurkevics: P-wave incidence angle (degrees from vertical)
        - s_arrival_time: S-wave arrival time (seconds from origin)
        - datetime: Event origin time (UTCDateTime compatible string)
    use_dynamic_params : bool, optional
        Whether to calculate and use dynamic windowing/filtering parameters
        based on spectral analysis (default=True)
        
    Returns:
    --------
    swspy.splitting object
        Splitting object ready for analysis with optimized parameters
    """
    
    # Get traces from event_data
    event_traces = event_data.get('traces', [])
    if not event_traces:
        raise ValueError("No traces in event_data")
    
    # Convert to stream if needed
    if isinstance(event_traces, list):
        stream = obspy.Stream(event_traces)
    else:
        stream = event_traces.copy()
    
    # Calculate dynamic parameters if requested
    if use_dynamic_params:
        print(f"  Calculating dynamic parameters...")
        dynamic_params = calculate_dynamic_parameters(event_data)
        print(f"    Dominant period: {dynamic_params['t_dom']:.3f} s")
        print(f"    Dynamic window length: {dynamic_params['dynamic_window_length']:.3f} s")
        print(f"    Optimal frequency range: {dynamic_params['optimal_freq_min']:.1f}-{dynamic_params['optimal_freq_max']:.1f} Hz")
        
        freq_min = dynamic_params['optimal_freq_min']
        freq_max = dynamic_params['optimal_freq_max']
        window_length = dynamic_params['dynamic_window_length']
        t_dom = dynamic_params['t_dom']
    else:
        # Use default fixed parameters
        freq_min = 5.0
        freq_max = 40.0
        window_length = 3.0
        t_dom = 0.1
        dynamic_params = None
    
    # Apply optimal filtering to the stream
    print(f"  Applying bandpass filter: {freq_min:.1f}-{freq_max:.1f} Hz")
    stream_filtered = stream.copy()
    stream_filtered.filter("bandpass", freqmin=freq_min, freqmax=freq_max)
    
    # Extract required metadata from event_data
    station_name = event_data.get('station', 'UNKNOWN')
    back_azimuth = event_data.get('back_azimuth')
    incidence_angle = event_data.get('incidence_eigenvalue_jurkevics')
    
    # Validate required fields
    if back_azimuth is None or np.isnan(back_azimuth):
        raise ValueError(f"Missing or invalid back_azimuth for station {station_name}")
    if incidence_angle is None or np.isnan(incidence_angle):
        raise ValueError(f"Missing or invalid incidence angle for station {station_name}")
    
    # Calculate S-arrival absolute time
    event_time = UTCDateTime(event_data['datetime'])
    s_arrival_time = float(event_data['s_arrival_time'])
    s_arrival_absolute = event_time + s_arrival_time
    p_arrival_time = float(event_data['p_arrival_time'])
    p_arrival_absolute = event_time + p_arrival_time
    
    print(f"  Creating SWSPy splitting object...")
    print(f"    Station: {station_name}")
    print(f"    Back-azimuth: {back_azimuth:.2f}°")
    print(f"    Incidence: {incidence_angle:.2f}°")
    print(f"    S-arrival: {s_arrival_absolute}")
    
    # Create splitting object with SWSPy using exact pattern from user
    # origin_times = [UTCDateTime(event_data['datetime'])]
    #P_phase_arrival_times=[p_arrival_absolute],

    splitting_event = swspy.splitting.create_splitting_object(
        stream_filtered, 
        stations_in=[station_name],
        back_azis_all_stations=[back_azimuth],
        receiver_inc_angles_all_stations=[incidence_angle],
        S_phase_arrival_times=[s_arrival_absolute]
    )
    #origin_event_times=[event_time], P_phase_arrival_times =[p_arrival_absolute]
    
    # Set dynamic analysis parameters on the splitting object
    if use_dynamic_params:
        # Dynamic windowing based on dominant period
        window_half_length = window_length / 2.0

        # MFAST-style windowing parameters
        splitting_event.toffbeg = window_half_length
        splitting_event.toffend = window_half_length
        # Number of windows (begin/end). Keep symmetric by default.
        n_windows = max(5, int(20 / t_dom))
        splitting_event.Nw_beg = n_windows
        splitting_event.Nw_end = n_windows
        # Time-step between candidate window offsets (seconds)
        dt_step = max(0.005, min(0.05, t_dom / 10.0))
        splitting_event.dTbeg = dt_step
        splitting_event.dTend = dt_step

        # Also set other analysis params
        splitting_event.rotate_step_deg = 1.0
        splitting_event.max_t_shift_s = min(0.2, t_dom)  # Max shift = dominant period, capped at 0.2s
        
        # Store dominant frequency for _setup_mfast_windowing
        splitting_event.T_mid = t_dom  # Store as T_mid (period)
        splitting_event.fs = stream_filtered[0].stats.sampling_rate

        print(f"  Dynamic analysis parameters set:")
        print(f"    Window pre/post S-pick (toffbeg/toffend): {splitting_event.toffbeg:.3f}s")
        print(f"    Max time shift: {splitting_event.max_t_shift_s:.3f}s")
        print(f"    Number of windows: {n_windows}")
    else:
                # Set T_mid for MFAST windowing
        #splitting_event.T_mid = t_dom
        #splitting_event.fs = stream_filtered[0].stats.sampling_rate
        
        # Use default fixed parameters (MFAST-style)
        #splitting_event.toffbeg = 0.3
        #splitting_event.toffend = splitting_event.T_mid / 2
        #splitting_event.dTbeg = 0.2
        #splitting_event.dTend = 0.08
        #splitting_event.Nw_beg = 5
        #splitting_event.Nw_end = int(1.5 * splitting_event.T_mid / splitting_event.dTend)
        #splitting_event.rotate_step_deg = 1.0
        #splitting_event.max_t_shift_s = 0.1

        splitting_event.overall_win_start_pre_fast_S_pick = 0.1
        splitting_event.overall_win_start_post_fast_S_pick = 0.3
        splitting_event.win_S_pick_tolerance = 0.0509
        splitting_event.rotate_step_deg = 2.0
        splitting_event.max_t_shift_s = 0.25
        splitting_event.n_win = 10
    
    
    # Store dynamic parameters in the splitting object for reference
    if dynamic_params is not None:
        splitting_event.dynamic_params = dynamic_params
    
    return splitting_event


In [ ]:
def extract_and_compare_dominant_periods(organized_waveforms, use_dynamic_params=False):
    """
    Extract T_mid from SWSPy splitting objects and compare with Baillard's dominant period.
    
    Creates splitting objects for each event to extract the dominant period (T_mid)
    calculated by SWSPy's FFT-based method, then compares with the Welch-based 
    dominant period from Baillard's method already stored in organized_waveforms.
    
    Parameters:
    -----------
    organized_waveforms : dict
        Dictionary with event IDs as keys, containing event data with traces,
        metadata, and 'dominant_period_baillard' already calculated
    use_dynamic_params : bool
        Whether to use dynamic filtering parameters when creating splitting objects
        (default=False to keep T_mid calculation consistent)
        
    Returns:
    --------
    dict
        Updated organized_waveforms with 'dominant_period_swspy' added to each event
    """
    
    print(f"Extracting T_mid from SWSPy splitting objects for {len(organized_waveforms)} events...")
    print("="*60)
    
    success_count = 0
    failed_events = []
    
    for event_id, event_data in organized_waveforms.items():
        try:
            # Create splitting object to extract T_mid (without running full analysis)
            splitting_obj = create_splitting_analysis(event_data, use_dynamic_params=use_dynamic_params)
            
            # Extract T_mid from splitting object
            t_mid_swspy = splitting_obj.Tmid
            
            # Store in event_data
            event_data['dominant_period_swspy'] = t_mid_swspy
            
            success_count += 1
            
            # Print progress every 10 events
            if success_count % 10 == 0:
                print(f"  Processed {success_count}/{len(organized_waveforms)} events...")
                
        except Exception as e:
            print(f"  ✗ Failed for event {event_id}: {e}")
            event_data['dominant_period_swspy'] = np.nan
            failed_events.append(event_id)
    
    return organized_waveforms

In [ ]:
organized_waveforms

In [ ]:
# Extract T_mid from SWSPy and compare with Baillard's dominant period
organized_waveforms = extract_and_compare_dominant_periods(organized_waveforms, use_dynamic_params=False)

In [ ]:
# Extract T_mid and Baillard's dominant period for comparison
comparison_data = []
for event_id, event_data in organized_waveforms.items():
    if event_data['dominant_period_baillard'] == 0:
        continue
    t_mid_swspy = event_data.get('dominant_period_swspy', np.nan)
    t_dom_baillard = event_data.get('dominant_period_baillard', np.nan) * 0.005
    comparison_data.append((event_id, t_mid_swspy, t_dom_baillard))

# Plot histogram of T_mid (SWSPy) and dominant_period_baillard
t_mid_values = [d[1] for d in comparison_data if not np.isnan(d[1])]
t_dom_baillard_values = [d[2] for d in comparison_data if not np.isnan(d[2])]

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(t_mid_values, bins=10, color='lightcoral', edgecolor='black')
plt.title('Histogram of Dominant Period (SWSPy)')
plt.xlabel('Dominant Period (s)')
plt.ylabel('Number of Events')
plt.xlim(0, 0.65)
plt.ylim(0, 60)
plt.grid(True)

plt.subplot(1, 2, 2)
plt.hist(t_dom_baillard_values, bins=40, color='skyblue', edgecolor='black')
plt.title('Histogram of Dominant Period (Baillard)')
plt.xlabel('Dominant Period (s)')
plt.ylabel('Number of Events')
plt.xlim(0, 0.65)
plt.ylim(0, 60)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Print mean, median, and range for both sets of dominant periods
if t_mid_values:
    print(f"SWSPy Dominant Period:")
    print(f"  Mean: {np.mean(t_mid_values):.3f} s")
    print(f"  Median: {np.median(t_mid_values):.3f} s")
    print(f"  Range: {np.min(t_mid_values):.3f} s to {np.max(t_mid_values):.3f} s")

if t_dom_baillard_values:
    print(f"\nBaillard Dominant Period:")
    print(f"  Mean: {np.mean(t_dom_baillard_values):.3f} s")
    print(f"  Median: {np.median(t_dom_baillard_values):.3f} s")
    print(f"  Range: {np.min(t_dom_baillard_values):.3f} s to {np.max(t_dom_baillard_values):.3f} s")

In [ ]:
# Filter t_dom_baillard_values > 0
t_dom_baillard_values_filtered = [v for v in t_dom_baillard_values if not np.isnan(v) and v > 0]

In [ ]:
if t_dom_baillard_values_filtered:
    print(f"\nBaillard Dominant Period (filtered > 0):")
    print(f"  Mean: {np.mean(t_dom_baillard_values_filtered):.3f} s")
    print(f"  Median: {np.median(t_dom_baillard_values_filtered):.3f} s")
    print(f"  Range: {np.min(t_dom_baillard_values_filtered):.3f} s to {np.max(t_dom_baillard_values_filtered):.3f} s")

In [ ]:
organized_waveforms[41212]

In [ ]:
# Now let's plot the difference between the S and P arrival times as a histogram
s_p_diff = []

for eid in organized_waveforms:
    s_p_diff.append(organized_waveforms[eid]['s_arrival_time'] - organized_waveforms[eid]['p_arrival_time'])

In [ ]:
plt.hist(s_p_diff, bins=20, color='lightgreen', edgecolor='black')
plt.title('Histogram of S-P Arrival Time Difference')
plt.xlabel('S-P Time Difference (s)')
plt.ylabel('Number of Events')
plt.grid(True)
plt.show()

In [ ]:
# Show mean, median, and range of s_p_diff
s_p_diff_filtered = [v for v in s_p_diff if not np.isnan(v) and v > 0]
if s_p_diff_filtered:
    print(f"S-P Arrival Time Difference:")
    print(f"  Mean: {np.mean(s_p_diff_filtered):.3f} s")
    print(f"  Median: {np.median(s_p_diff_filtered):.3f} s")
    print(f"  Range: {np.min(s_p_diff_filtered):.3f} s to {np.max(s_p_diff_filtered):.3f} s")

In [ ]:
# Perform one shear-wave split using the updated windowing code, to ensure it works
passing_waveforms_first = {41214: passing_waveforms[41214]}

In [ ]:
passing_waveforms_first

In [ ]:
def perform_splitting_on_organized_waveforms(organized_waveforms, use_dynamic_params=False, mode='swspy', plot_results=False):
    """
    Perform shear-wave splitting analysis on all events in organized_waveforms.
    
    This function assumes organized_waveforms has been filtered by apply_quality_control()
    and contains only events that pass QC thresholds. It extracts horizontal components
    and performs splitting analysis using the back_azimuth for coordinate rotation.
    
    Parameters:
    -----------
    organized_waveforms : dict
        Dictionary with event IDs as keys, containing QC-filtered traces and metadata
        Must have: traces, back_azimuth, station, and all other event metadata

    mode : str
        Splitting analysis method to use: 'swspy' or 'baillard' (default: 'swspy')
        
    Returns:
    --------
    dict
        Dictionary with event IDs as keys, each containing:
        {
            event_id: {
                'result': result_dict,  # Splitting parameters and metadata
                'splitting_obj': splitting_obj  # SWSPy splitting object
            },
            ...
        }
    """
    
    print(f"\n{'='*60}")
    print("Performing Shear-Wave Splitting Analysis")
    print(f"{'='*60}")
    print(f"Processing {len(organized_waveforms)} QC-filtered events...")
    
    event_results = {}
    
    # Track statistics
    stats = {
        'total_events': len(organized_waveforms),
        'missing_components': 0,
        'missing_back_azimuth': 0,
        'splitting_errors': 0,
        'successful_splits': 0
    }
    
    for event_id, event_data in organized_waveforms.items():
        print(f"\n{'─'*60}")
        print(f"Event {event_id}")
        print(f"{'─'*60}")
        
        # Get traces
        event_traces = event_data.get('traces', [])
        if not event_traces:
            print(f"  ✗ No traces found")
            stats['missing_components'] += 1
            continue
        
        # Convert to stream if needed
        if isinstance(event_traces, list):
            event_stream = obspy.Stream(event_traces)
        else:
            event_stream = event_traces
        
        # Find N and E components
        trace_n = None
        trace_e = None
        trace_z = None
        
        for tr in event_stream:
            component = tr.stats.channel[-1].upper()
            if component in ['N', '1']:
                trace_n = tr
            elif component in ['E', '2']:
                trace_e = tr
            elif component == 'Z':
                trace_z = tr
        
        # Check if we have horizontal components
        if trace_n is None or trace_e is None:
            print(f"  ✗ Missing horizontal components (N: {trace_n is not None}, E: {trace_e is not None})")
            stats['missing_components'] += 1
            continue
        
        print(f"  ✓ Found horizontal components")
        
        # Check for back-azimuth
        back_azimuth = event_data.get('back_azimuth')
        if back_azimuth is None or np.isnan(back_azimuth):
            print(f"  ✗ Missing back-azimuth")
            stats['missing_back_azimuth'] += 1
            continue
        
        print(f"  ✓ Back-azimuth: {back_azimuth:.2f}°")
        
        # Get station and other metadata
        station_name = event_data.get('station', 'UNKNOWN')
        magnitude = event_data.get('magnitude', np.nan)
        snr_horizontal = event_data.get('snr_horizontal', np.nan)
        rectilinearity = event_data.get('rectilinearity_jurkevics', np.nan)
        incidence = event_data.get('incidence_eigenvalue_jurkevics', np.nan)
        
        print(f"  Station: {station_name}")
        print(f"  Magnitude: {magnitude:.1f}")
        print(f"  SNR: {snr_horizontal:.2f}")
        print(f"  Rectilinearity: {rectilinearity:.3f}")
        print(f"  Incidence: {incidence:.1f}°")
        
        # Perform splitting analysis using Baillard method
        print(f"\n  → Running Baillard splitting analysis...")
        
        try:
           
            if mode=='baillard':
                # Call Baillard splitting function
                splitting_result = perform_splitting_analysis_baillard(
                    event_data,
                    s_window=[0.02,0.3],
                    min_lag=0,
                    max_lag=60,
                    Nlags=60,
                    Nangles=90,
                    flag_adapt_window=True,
                    flag_adapt_maxlag=True,
                    min_thres=0.5,
                    min_numbers=2,
                    plot_results=plot_results
                )
                
                # Baillard returns dict only (no splitting_obj like SWSPy)
                splitting_obj = None

            # Hemmett-adapted SWSPy-like implementation that draws on MFAST
            elif mode=='swspy':
                # Call SWSPy-like splitting function
                splitting_result, splitting_obj = perform_splitting_analysis(
                    event_data, use_dynamic_params=use_dynamic_params, plot_results=plot_results
                )

            elif mode=='teanby_baillard':
                # Call Teanby clustering with Baillard method
                splitting_result = tbc.teanby_clustering_analysis(
                    event_data,
                    T_beg_1=0.02, T_end_0=0.3,
                    dT_beg=0.005, dT_end=0.005,
                    N_beg=10, N_end=10,
                    min_lag=0, max_lag=60,
                    Nlags=60, Nangles=90,
                    flag_adapt_window=True,
                    flag_adapt_maxlag=True,
                    min_thres=0.5,
                    min_numbers=2,
                    plot_results=plot_results,
                    output_dir=None,
                    N_c_min=5
                )
                splitting_obj = None

            if splitting_result.get('success', False):
                # Add additional metadata to result
                splitting_result['event_id'] = event_id
                splitting_result['back_azimuth'] = back_azimuth
                splitting_result['snr_horizontal'] = snr_horizontal
                splitting_result['rectilinearity_jurkevics'] = rectilinearity
                splitting_result['incidence_eigenvalue_jurkevics'] = incidence
                splitting_result['event_lat'] = event_data.get('latitude')
                splitting_result['event_lon'] = event_data.get('longitude')
                splitting_result['event_depth'] = event_data.get('depth')
                splitting_result['event_datetime'] = event_data.get('datetime')
                
                # Store both result dict and splitting object for later use
                event_results[event_id] = {
                    'result': splitting_result,
                    'splitting_obj': splitting_obj
                }
                stats['successful_splits'] += 1
                
                print(f"  ✓ SUCCESS!")
                if not np.isnan(splitting_result['phi']):
                    print(f"    Fast axis (φ): {splitting_result['phi']:.1f}°")
                else:
                    print(f"    Fast axis (φ): N/A")
                    
                if not np.isnan(splitting_result['dt']):
                    print(f"    Delay time (δt): {splitting_result['dt']:.3f}s")
                else:
                    print(f"    Delay time (δt): N/A")
                    
                if not np.isnan(splitting_result.get('phi_error', np.nan)):
                    print(f"    φ error: ±{splitting_result['phi_error']:.1f}°")
                if not np.isnan(splitting_result.get('dt_error', np.nan)):
                    print(f"    δt error: ±{splitting_result['dt_error']:.3f}s")
                if 'dominant_period' in splitting_result:
                    print(f"    Dominant period: {splitting_result['dominant_period']:.3f}s")
            else:
                error_msg = splitting_result.get('error', 'Unknown error')
                print(f"  ✗ FAILED: {error_msg}")
                if 'traceback' in splitting_result:
                    print(f"  Traceback:\n{splitting_result['traceback']}")
                stats['splitting_errors'] += 1
                
        except Exception as e:
            import traceback
            print(f"  ✗ ERROR during splitting: {e}")
            print(f"  Traceback:\n{traceback.format_exc()}")
            stats['splitting_errors'] += 1
            continue
    
    # Print summary
    print(f"\n{'='*60}")
    print("Splitting Analysis Summary")
    print(f"{'='*60}")
    print(f"Total events processed: {stats['total_events']}")
    print(f"Successful splits: {stats['successful_splits']} ({100*stats['successful_splits']/stats['total_events']:.1f}%)")
    print(f"\nFailure breakdown:")
    print(f"  Missing components: {stats['missing_components']}")
    print(f"  Missing back-azimuth: {stats['missing_back_azimuth']}")
    print(f"  Splitting errors: {stats['splitting_errors']}")
    
    # Calculate splitting parameter statistics if we have results
    if event_results:

        if mode!='swspy':
            # Extract phi and dt values from nested structure
            phi_values = [r['result']['phi_rad'] for r in event_results.values() 
                        if not np.isnan(r['result']['phi_rad'])]
            dt_values = [r['result']['dt'] for r in event_results.values() 
                        if not np.isnan(r['result']['dt'])]
        
        else:
            # Extract phi and dt values from nested structure
            phi_values = [r['result']['phi'] for r in event_results.values() 
                        if not np.isnan(r['result']['phi'])]
            dt_values = [r['result']['dt'] for r in event_results.values() 
                        if not np.isnan(r['result']['dt'])]
        
        if phi_values and dt_values:
            print(f"\n{'─'*60}")
            print("Splitting Parameter Statistics")
            print(f"{'─'*60}")
            print(f"Fast axis direction (φ):")
            print(f"  Mean: {np.mean(phi_values):.1f}° ± {np.std(phi_values):.1f}°")
            print(f"  Range: {np.min(phi_values):.1f}° to {np.max(phi_values):.1f}°")
            print(f"  Median: {np.median(phi_values):.1f}°")
            
            print(f"\nDelay time (δt):")
            print(f"  Mean: {np.mean(dt_values):.3f} ± {np.std(dt_values):.3f}s")
            print(f"  Range: {np.min(dt_values):.3f}s to {np.max(dt_values):.3f}s")
            print(f"  Median: {np.median(dt_values):.3f}s")
            
            # Add to stats
            stats['phi_mean'] = float(np.mean(phi_values))
            stats['phi_std'] = float(np.std(phi_values))
            stats['dt_mean'] = float(np.mean(dt_values))
            stats['dt_std'] = float(np.std(dt_values))
        else:
            print(f"\n  Note: No valid splitting parameters for statistics")
    
    # Return the event_results dictionary directly
    # Each entry contains both 'result' and 'splitting_obj'
    return event_results

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_baillard = perform_splitting_on_organized_waveforms(passing_waveforms, use_dynamic_params=False, mode='baillard', plot_results=False)

In [ ]:
passing_waveforms

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_swspy = perform_splitting_on_organized_waveforms(passing_waveforms, mode='swspy', plot_results=True, first_window_start = 2, last_window_start = 1, first_window_end = 1.5, last_window_end = 2.5,
                                                         n_win = 10, s_pick_uncertainty = 0.0509)

In [ ]:
passing_waveforms_first[41214]

In [ ]:
passing_waveforms_first[41214]['origin_time'] - passing_waveforms_first[41214]['traces'][0].stats.starttime


In [ ]:
# Plot waveforms for the first event from passing_waveforms
if len(passing_waveforms) > 0:
    # Get first event from passing_waveforms
    first_event_id = list(passing_waveforms.keys())[1]
    event_data = passing_waveforms[first_event_id]
    
    # Get traces
    st = event_data['traces'].copy()
    print(f"Plotting first event from passing_waveforms: Event ID {first_event_id}")
    print(f"Station: {event_data['station']}")
    
    # Sort traces by channel (Z, N, E order)
    st = st.sort(['channel'])
    
    # Create figure
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # Calculate time offset between trace start and origin time
    event_origin = UTCDateTime(event_data['datetime'])
    trace_start = st[0].stats.starttime
    time_offset = float(event_origin - trace_start)  # seconds between trace start and origin
    
    # P and S arrivals are relative to origin_time
    p_time_rel_origin = event_data['p_arrival_time'] + time_offset
    s_time_rel_origin = event_data['s_arrival_time'] + time_offset
    
    # Debug: print timing information
    print(f"  time_offset (origin - trace_start): {time_offset:.3f} s")
    print(f"  P-arrival (rel to origin): {p_time_rel_origin:.3f} s")
    print(f"  S-arrival (rel to origin): {s_time_rel_origin:.3f} s")
    
    # Calculate time window: origin_time (0) to 0.3s after S-pick
    time_start = time_offset + 1  # Start at origin time
    #time_end = s_time_rel_origin + time_offset + 1
    time_end = s_time_rel_origin + time_offset + 2.5
    
    print(f"  Plot window: [{time_start:.3f}, {time_end:.3f}] s")
    
    # Plot each component
    for i, tr in enumerate(st):
        ax = axes[i]
        
        # Time array relative to origin_time (not trace start)
        time = (np.arange(len(tr.data)) / tr.stats.sampling_rate) + time_offset
        
        # Debug: print time range for first trace
        if i == 0:
            print(f"  Time array range: [{time[0]:.3f}, {time[-1]:.3f}] s")
            print(f"  Data points in plot window: {np.sum((time >= time_start) & (time <= time_end))}")
        
        # Plot waveform
        ax.plot(time, tr.data, 'k', linewidth=0.8)
        ax.set_ylabel(f"{tr.stats.channel}\nAmplitude", fontweight='bold', fontsize=11)
        ax.grid(True, alpha=0.3)
        
        # Normalize to show all traces clearly
        ylim = np.max(np.abs(tr.data))
        ax.set_ylim([-ylim, ylim])
        
        # Set x-limits: origin_time (0) to 0.3s after S-pick
        ax.set_xlim([time_start, time_end])
        
        
        # Add phase markers
        if p_time_rel_origin > 0:
            ax.axvline(p_time_rel_origin + time_offset, color='blue', linestyle='--', alpha=0.8, 
                      linewidth=2, label='P-wave', zorder=5)
        if s_time_rel_origin > 0:
            ax.axvline(s_time_rel_origin + time_offset, color='red', linestyle='--', alpha=0.8, 
                      linewidth=2, label='S-wave', zorder=5)
            
                # Add S-wave window shading (0.3s window after S-pick)
        ax.axvspan(s_time_rel_origin + time_offset, s_time_rel_origin + time_offset + 2*0.085, 
                  alpha=0.15, color='red', zorder=0, label='S-wave Window' if i == 0 else '')
        
        if i == 0:
            ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
        
        # Remove top and right spines for cleaner look
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    axes[-1].set_xlabel('Time since origin (s)', fontweight='bold', fontsize=11)
    
    # Create title with event information
    title = (f"Three-Component Seismogram (Filtered 5-40 Hz) - Station {event_data['station']}\n"
            f"Event ID: {first_event_id}, Origin: {event_data['datetime']}, "
            f"Depth: {event_data['depth']:.2f} km, Magnitude: {event_data['magnitude']:.1f}")
    
    fig.suptitle(title, fontsize=12, fontweight='bold', y=0.98)
    
    # Add text box with event parameters
    #info_text = (f"Event Parameters:\n"
    #            f"  Back-azimuth: {event_data['back_azimuth']:.1f}°\n"
    #            f"  Incidence: {event_data['incidence_eigenvalue_jurkevics']:.1f}°\n"
    #            f"  SNR: {event_data['snr_horizontal']:.2f}\n"
    #            f"  Rectilinearity: {event_data['rectilinearity_jurkevics']:.3f}\n"
    #            f"  Window: Origin to S+0.3s")
    
    #axes[0].text(0.98, 0.97, info_text, transform=axes[0].transAxes,
    #            fontsize=9, verticalalignment='top', horizontalalignment='right',
    #            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9, edgecolor='black', linewidth=1.5))
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nEvent ID: {first_event_id}")
    print(f"  Trace starts at: {time_offset:.3f} s before origin")
    print(f"  P-arrival at: {p_time_rel_origin:.3f} s (relative to origin)")
    print(f"  S-arrival at: {s_time_rel_origin:.3f} s (relative to origin)")
    print(f"  Window shown: {time_start:.3f} to {time_end:.3f} s (origin to S+0.3s)")
    print(f"{'='*80}\n")

else:
    print("No events available in passing_waveforms")

In [ ]:
# Plot waveforms for the first event from passing_waveforms
if len(passing_waveforms) > 0:
    # Get first event from passing_waveforms
    first_event_id = list(passing_waveforms.keys())[0]
    event_data = passing_waveforms[first_event_id]
    
    # Get traces
    st = event_data['traces'].copy()
    st = st.select(channel='*N') + st.select(channel='*E')
    print(f"Plotting first event from passing_waveforms: Event ID {first_event_id}")
    print(f"Station: {event_data['station']}")
    
    # Sort traces by channel (Z, N, E order)
    st = st.sort(['channel'])
    
    # Create figure
    fig, axes = plt.subplots(2, 1, figsize=(14*6, 10*6), sharex=True)
    
    # Calculate time offset between trace start and origin time
    event_origin = UTCDateTime(event_data['datetime'])
    trace_start = st[0].stats.starttime
    time_offset = float(event_origin - trace_start)  # seconds between trace start and origin
    
    # P and S arrivals are relative to origin_time
    p_time_rel_origin = event_data['p_arrival_time'] + time_offset
    s_time_rel_origin = event_data['s_arrival_time'] + time_offset
    
    # Debug: print timing information
    print(f"  time_offset (origin - trace_start): {time_offset:.3f} s")
    print(f"  P-arrival (rel to origin): {p_time_rel_origin:.3f} s")
    print(f"  S-arrival (rel to origin): {s_time_rel_origin:.3f} s")
    
    # Calculate time window: origin_time (0) to 0.3s after S-pick
    time_start = time_offset + 1  # Start at origin time
    time_end = s_time_rel_origin + time_offset + 1
    Nbeg= 10
    Nend = 10

    Terr = 0.0509
    
    Tbeg0 = 1 * Terr
    Tbeg1 = 2 * Terr
    dTbeg = (Tbeg1 - Tbeg0) / Nbeg

    # Calc window start times
    start_times = []
    for i in range(0,Nbeg+1):
        start_times.append(s_time_rel_origin + time_offset - Tbeg1 + i * dTbeg)

    #t_mid = event_data['dominant_period_swspy']
    t_mid = 0.07
    
    Tend0 = t_mid * (1.5)

    Tend1 = t_mid * (2.5)

    dTend = (Tend1 - Tend0) / Nend

    # Calc window end times
    end_times = []
    for i in range(0, Nend+1):
        end_times.append(s_time_rel_origin + time_offset + Tend0 + i * dTend)
    
    print(f"  Plot window: [{time_start:.3f}, {time_end:.3f}] s")
    
    # Plot each component
    # Filter stream to just be N, E
    st = st.select(channel='*N') + st.select(channel='*E')
    for i, tr in enumerate(st):
        ax = axes[i]
        
        # Time array relative to origin_time (not trace start)
        time = (np.arange(len(tr.data)) / tr.stats.sampling_rate) + time_offset
        
        # Debug: print time range for first trace
        if i == 0:
            print(f"  Time array range: [{time[0]:.3f}, {time[-1]:.3f}] s")
            print(f"  Data points in plot window: {np.sum((time >= time_start) & (time <= time_end))}")
        
        # Plot waveform
        ax.plot(time, tr.data, 'k', linewidth=10)
        ax.set_ylabel(f"{tr.stats.channel}\nAmplitude", fontweight='bold', fontsize=250)
        ax.grid(True, alpha=0.3)
        
        # Normalize to show all traces clearly
        ylim = np.max(np.abs(tr.data))
        ax.set_ylim([-ylim, ylim])
        
        # Set x-limits: origin_time (0) to 0.3s after S-pick
        ax.set_xlim([time_start, time_end])
        

        dom_period_start = s_time_rel_origin + time_offset
        dom_period_end = s_time_rel_origin + time_offset + t_mid
        #ax.axvspan(dom_period_start, dom_period_end, alpha=0.15, color='orange', zorder=0, label=f'Dominant period: {t_mid:.3f}s')

        # Add phase markers
        if p_time_rel_origin > 0:
            ax.axvline(p_time_rel_origin + time_offset, color='blue', linestyle='--', alpha=0.8, 
                      linewidth=10, label='P-wave', zorder=5)
        if s_time_rel_origin > 0:
            ax.axvline(s_time_rel_origin + time_offset, color='red', linestyle='--', alpha=0.8, 
                      linewidth=10, label='S-wave', zorder=5)
            
        #ax.axvline(t_mid * (5/6) + 0.15 + s_time_rel_origin + time_offset, color='orange', linestyle='--', 
        #           alpha=1.0, linewidth=2, label='Tend0 = 5/6 Tmid + 0.15', zorder=5)
        
        #ax.axvline(t_mid * (2.5) + 0.15 + s_time_rel_origin + time_offset, color='black', linestyle='--', 
        #           alpha=0.8, linewidth=2, label='Tend1 = 2.5 Tmid + 0.15', zorder=5)

         # Add window start times visualization (before S-pick)
        # Light gray box from first to last start time
        ax.axvspan(start_times[0], start_times[Nbeg], 
                  alpha=0.3, color='gray', zorder=0, label='Window start range: 2-1σ' if i == 0 else '')
        # Individual start time lines
        for start_time in start_times:
            ax.axvline(start_time, color='gray', linestyle=':', alpha=0.7, linewidth=8, zorder=1)

        # Add window end times visualization (after S-pick)
        # Light gray box from first to last end time
        ax.axvspan(end_times[0], end_times[Nend], 
                  alpha=0.3, color='gray', zorder=0, label='Window end range: 1.5-2.5 Tmid' if i == 0 else '')
        # Individual end time lines
        for end_time in end_times:
            ax.axvline(end_time, color='gray', linestyle=':', alpha=0.7, linewidth=8, zorder=1)
        
        #ax.axvline(s_time_rel_origin + time_offset - (s_time_rel_origin - p_time_rel_origin)/2 , color='green', linestyle='--', 
        #           alpha=0.8, linewidth=2, label='(S-P) / 2', zorder=5)

        #ax.axvline(s_time_rel_origin + time_offset - Tbeg0, color='purple', linestyle='--', 
        #           alpha=0.8, linewidth=2, label='Tbeg0 = 0.1 or ((S-P) / 2) / (Nbeg + 1)', zorder=5)
        
        #ax.axvline(s_time_rel_origin + time_offset - Tbeg1, color='brown', linestyle='--', 
        #           alpha=0.8, linewidth=2, label='Tbeg1 = Tbeg0 + (Nbeg * dTbeg)', zorder=5)
        
        
        if i == 0:
            ax.legend(loc='upper left', fontsize=100, framealpha=0.9)
        
        # Remove top and right spines for cleaner look
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(axis='both', which='major', labelsize=180)
    
    axes[-1].set_xlabel('Time since origin (s)', fontweight='bold', fontsize=250)

    
    # Create title with event information
    #title = (f"Three-Component Seismogram (Filtered 5-40 Hz) - Station {event_data['station']}\n"
    #        f"Event ID: {first_event_id}, Origin: {event_data['datetime']}, "
    #        f"Depth: {event_data['depth']:.2f} km, Magnitude: {event_data['magnitude']:.1f}")
    
    #fig.suptitle(title, fontsize=12, fontweight='bold', y=0.98)
    
    # Add text box with event parameters
    info_text = (f"Event Parameters:\n"
                f"  Back-azimuth: {event_data['back_azimuth']:.1f}°\n"
                f"  Incidence: {event_data['incidence_eigenvalue_jurkevics']:.1f}°\n"
                f"  SNR: {event_data['snr_horizontal']:.2f}\n"
                f"  Rectilinearity: {event_data['rectilinearity_jurkevics']:.3f}\n")
    
    axes[0].text(0.98, 0.97, info_text, transform=axes[0].transAxes,
                fontsize=100, verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9, edgecolor='black', linewidth=5))
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nEvent ID: {first_event_id}")
    print(f"  Trace starts at: {time_offset:.3f} s before origin")
    print(f"  P-arrival at: {p_time_rel_origin:.3f} s (relative to origin)")
    print(f"  S-arrival at: {s_time_rel_origin:.3f} s (relative to origin)")
    print(f"  Window shown: {time_start:.3f} to {time_end:.3f} s (origin to S+0.3s)")
    print(f"{'='*80}\n")

else:
    print("No events available in passing_waveforms")

In [ ]:
start_times

In [ ]:
end_times

In [ ]:
# Plot waveforms for the first event from passing_waveforms
if len(passing_waveforms) > 0:
    # Get first event from passing_waveforms
    first_event_id = list(passing_waveforms.keys())[0]
    event_data = passing_waveforms[first_event_id]
    
    # Get traces
    st = event_data['traces'].copy()
    print(f"Plotting first event from passing_waveforms: Event ID {first_event_id}")
    print(f"Station: {event_data['station']}")
    
    # Sort traces by channel (Z, N, E order)
    st = st.sort(['channel'])
    
    # Create figure
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # Calculate time offset between trace start and origin time
    event_origin = UTCDateTime(event_data['datetime'])
    trace_start = st[0].stats.starttime
    time_offset = float(event_origin - trace_start)  # seconds between trace start and origin
    
    # P and S arrivals are relative to origin_time
    p_time_rel_origin = event_data['p_arrival_time'] + time_offset
    s_time_rel_origin = event_data['s_arrival_time'] + time_offset
    
    # Debug: print timing information
    print(f"  time_offset (origin - trace_start): {time_offset:.3f} s")
    print(f"  P-arrival (rel to origin): {p_time_rel_origin:.3f} s")
    print(f"  S-arrival (rel to origin): {s_time_rel_origin:.3f} s")
    
    # Calculate time window: origin_time (0) to 0.3s after S-pick
    time_start = time_offset + 1  # Start at origin time
    time_end = s_time_rel_origin + time_offset + 1
    
    print(f"  Plot window: [{time_start:.3f}, {time_end:.3f}] s")
    
    # Plot each component
    for i, tr in enumerate(st):
        ax = axes[i]
        
        # Time array relative to origin_time (not trace start)
        time = (np.arange(len(tr.data)) / tr.stats.sampling_rate) + time_offset
        
        # Debug: print time range for first trace
        if i == 0:
            print(f"  Time array range: [{time[0]:.3f}, {time[-1]:.3f}] s")
            print(f"  Data points in plot window: {np.sum((time >= time_start) & (time <= time_end))}")
        
        # Plot waveform
        ax.plot(time, tr.data, 'k', linewidth=0.8)
        ax.set_ylabel(f"{tr.stats.channel}\nAmplitude", fontweight='bold', fontsize=11)
        ax.grid(True, alpha=0.3)
        
        # Normalize to show all traces clearly
        ylim = np.max(np.abs(tr.data))
        ax.set_ylim([-ylim, ylim])
        
        # Set x-limits: origin_time (0) to 0.3s after S-pick
        ax.set_xlim([time_start, time_end])
        
        # Add frequency analysis window (0.1 to 0.3 seconds after S-pick)
        freq_window_start = s_time_rel_origin + time_offset - 0.1
        freq_window_end = s_time_rel_origin + time_offset + 0.3
        ax.axvspan(freq_window_start, freq_window_end, 
                  alpha=0.15, color='purple', zorder=0, 
                  label='Frequency analysis window' if i == 0 else '')
        
        # Add phase markers
        if p_time_rel_origin > 0:
            ax.axvline(p_time_rel_origin + time_offset, color='blue', linestyle='--', alpha=0.8, 
                      linewidth=2, label='P-wave', zorder=5)
        if s_time_rel_origin > 0:
            ax.axvline(s_time_rel_origin + time_offset, color='red', linestyle='--', alpha=0.8, 
                      linewidth=2, label='S-wave', zorder=5)
        
        if i == 0:
            ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
        
        # Remove top and right spines for cleaner look
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    axes[-1].set_xlabel('Time since origin (s)', fontweight='bold', fontsize=11)
    
    # Create title with event information
    title = (f"Three-Component Seismogram (Filtered 5-40 Hz) - Station {event_data['station']}\n"
            f"Event ID: {first_event_id}, Origin: {event_data['datetime']}, "
            f"Depth: {event_data['depth']:.2f} km, Magnitude: {event_data['magnitude']:.1f}")
    
    fig.suptitle(title, fontsize=12, fontweight='bold', y=0.98)
    
    # Add text box with event parameters
    info_text = (f"Event Parameters:\n"
                f"  Back-azimuth: {event_data['back_azimuth']:.1f}°\n"
                f"  Incidence: {event_data['incidence_eigenvalue_jurkevics']:.1f}°\n"
                f"  SNR: {event_data['snr_horizontal']:.2f}\n"
                f"  Rectilinearity: {event_data['rectilinearity_jurkevics']:.3f}\n"
                f"  Freq. window: S-0.1s to S+0.3s")
    
    axes[0].text(0.98, 0.97, info_text, transform=axes[0].transAxes,
                fontsize=9, verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9, edgecolor='black', linewidth=1.5))
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nEvent ID: {first_event_id}")
    print(f"  Trace starts at: {time_offset:.3f} s before origin")
    print(f"  P-arrival at: {p_time_rel_origin:.3f} s (relative to origin)")
    print(f"  S-arrival at: {s_time_rel_origin:.3f} s (relative to origin)")
    print(f"  Window shown: {time_start:.3f} to {time_end:.3f} s (origin to S+0.3s)")
    print(f"{'='*80}\n")

else:
    print("No events available in passing_waveforms")

In [ ]:
# Plot waveforms for the first event from passing_waveforms
if len(passing_waveforms) > 0:
    # Get first event from passing_waveforms
    first_event_id = list(passing_waveforms.keys())[0]
    event_data = passing_waveforms[first_event_id]
    
    # Get traces
    st = event_data['traces'].copy()
    print(f"Plotting first event from passing_waveforms: Event ID {first_event_id}")
    print(f"Station: {event_data['station']}")
    
    # Sort traces by channel (Z, N, E order)
    st = st.sort(['channel'])
    
    # Create figure
    fig, axes = plt.subplots(3, 1, figsize=(14*6, 10*6), sharex=True)
    
    # Calculate time offset between trace start and origin time
    event_origin = UTCDateTime(event_data['datetime'])
    trace_start = st[0].stats.starttime
    time_offset = float(event_origin - trace_start)  # seconds between trace start and origin
    
    # P and S arrivals are relative to origin_time
    p_time_rel_origin = event_data['p_arrival_time'] + time_offset
    s_time_rel_origin = event_data['s_arrival_time'] + time_offset
    
    # Debug: print timing information
    print(f"  time_offset (origin - trace_start): {time_offset:.3f} s")
    print(f"  P-arrival (rel to origin): {p_time_rel_origin:.3f} s")
    print(f"  S-arrival (rel to origin): {s_time_rel_origin:.3f} s")
    
    # Calculate time window: origin_time (0) to 0.3s after S-pick
    time_start = time_offset + 1  # Start at origin time
    time_end = s_time_rel_origin + time_offset + 1
    
    print(f"  Plot window: [{time_start:.3f}, {time_end:.3f}] s")
    
    # Extract unique window start and end times from the data
    # Start times (before S-pick): 0.01, 0.04, 0.07, 0.10, 0.14 s
    # End times (after S-pick): 0.21 to 0.35 s
    window_start_times = [0.01, 0.04, 0.07, 0.10, 0.14]
    window_end_times = [0.21, 0.22, 0.23, 0.24, 0.25, 0.26, 0.27, 0.28, 0.29, 0.30, 0.31, 0.32, 0.33, 0.34, 0.35]
    
    # Convert to absolute times (relative to trace start)
    window_start_times_abs = [s_time_rel_origin + time_offset - t for t in window_start_times]
    window_end_times_abs = [s_time_rel_origin + time_offset + t for t in window_end_times]
    
    # Plot each component
    for i, tr in enumerate(st):
        ax = axes[i]
        
        # Time array relative to origin_time (not trace start)
        time = (np.arange(len(tr.data)) / tr.stats.sampling_rate) + time_offset
        
        # Debug: print time range for first trace
        if i == 0:
            print(f"  Time array range: [{time[0]:.3f}, {time[-1]:.3f}] s")
            print(f"  Data points in plot window: {np.sum((time >= time_start) & (time <= time_end))}")
        
        # Plot waveform
        ax.plot(time, tr.data, 'k', linewidth=0.8)
        ax.set_ylabel(f"{tr.stats.channel}\nAmplitude", fontweight='bold', fontsize=11)
        ax.grid(True, alpha=0.3)
        
        # Normalize to show all traces clearly
        ylim = np.max(np.abs(tr.data))
        ax.set_ylim([-ylim, ylim])
        
        # Set x-limits: origin_time (0) to 0.3s after S-pick
        ax.set_xlim([time_start, time_end])
        
        # Add window start times visualization (before S-pick)
        # Light gray box from first to last start time
        ax.axvspan(window_start_times_abs[-1], window_start_times_abs[0], 
                  alpha=0.1, color='gray', zorder=0, label='Window start range' if i == 0 else '')
        # Individual start time lines
        for start_time in window_start_times_abs:
            ax.axvline(start_time, color='gray', linestyle=':', alpha=0.5, linewidth=1, zorder=1)
        
        # Add window end times visualization (after S-pick)
        # Light gray box from first to last end time
        ax.axvspan(window_end_times_abs[0], window_end_times_abs[-1], 
                  alpha=0.1, color='gray', zorder=0, label='Window end range' if i == 0 else '')
        # Individual end time lines
        for end_time in window_end_times_abs:
            ax.axvline(end_time, color='gray', linestyle=':', alpha=0.5, linewidth=1, zorder=1)
        
        # Add phase markers
        if p_time_rel_origin > 0:
            ax.axvline(p_time_rel_origin + time_offset, color='blue', linestyle='--', alpha=0.8, 
                      linewidth=2, label='P-wave', zorder=5)
        if s_time_rel_origin > 0:
            ax.axvline(s_time_rel_origin + time_offset, color='red', linestyle='--', alpha=0.8, 
                      linewidth=2, label='S-wave', zorder=5)
        
        if i == 0:
            ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
        
        # Remove top and right spines for cleaner look
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    axes[-1].set_xlabel('Time since origin (s)', fontweight='bold', fontsize=11)
    
    # Create title with event information
    title = (f"Three-Component Seismogram (Filtered 5-40 Hz) - Station {event_data['station']}\n"
            f"Event ID: {first_event_id}, Origin: {event_data['datetime']}, "
            f"Depth: {event_data['depth']:.2f} km, Magnitude: {event_data['magnitude']:.1f}")
    
    fig.suptitle(title, fontsize=12, fontweight='bold', y=0.98)
    
    # Add text box with event parameters
    info_text = (f"Event Parameters:\n"
                f"  Back-azimuth: {event_data['back_azimuth']:.1f}°\n"
                f"  Incidence: {event_data['incidence_eigenvalue_jurkevics']:.1f}°\n"
                f"  SNR: {event_data['snr_horizontal']:.2f}\n"
                f"  Rectilinearity: {event_data['rectilinearity_jurkevics']:.3f}\n"
                f"  Window: Origin to S+0.3s")
    
    axes[0].text(0.98, 0.97, info_text, transform=axes[0].transAxes,
                fontsize=9, verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9, edgecolor='black', linewidth=1.5))
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nEvent ID: {first_event_id}")
    print(f"  Trace starts at: {time_offset:.3f} s before origin")
    print(f"  P-arrival at: {p_time_rel_origin:.3f} s (relative to origin)")
    print(f"  S-arrival at: {s_time_rel_origin:.3f} s (relative to origin)")
    print(f"  Window shown: {time_start:.3f} to {time_end:.3f} s (origin to S+0.3s)")
    print(f"{'='*80}\n")

else:
    print("No events available in passing_waveforms")

In [ ]:
passing_waveforms_first[41214]['origin_time']

In [ ]:
passing_waveforms_first[41214]['s_arrival_time']

In [ ]:
passing_waveforms_first[41214]['traces'][0].stats.starttime

In [ ]:
passing_waveforms_first[41214]['origin_time'] - passing_waveforms_first[41214]['traces'][0].stats.starttime + passing_waveforms_first[41214]['s_arrival_time']

In [ ]:
passing_waveforms

In [ ]:
catalog

In [ ]:
def plot_movehisto2d_time(self,y_param,minlambda_select='min', 
                              x_label='X',y_label='Y',title='',ax=None,
                              angle_mode='trigo',lag_mode='sample',sampling_rate=200,
                              window_unit=None,window_bottom=None,window_top=None,
                              **movehisto2d_kwargs):
        """
        Function made to use movehist_2d and show variations of splitting parameters with time
        parameters are similat to the one for movehisto2d
        Plots showing angles used trigo convention by default (ie CCW from East) however angles can
        be shown using azimuth conention by specifyin angle_mode='azimuth', in that case all paramters
        should be given using that convention (ie. y_width, y_start,y_cycle...)
        
        parameter should be given in degrees
        
        Input
        ----
            angle_mode: str: ['trigo','azimuth']
            lag_mode: str: ['sample','ms','s']
            sampling_rate: float: sampling rate of the data to convert lags from samples to ms
        """

        
        ### Retrieve proper parameters
        
        elems_dic=self.get_dic(minlambda_select=minlambda_select)
        
        ### Select arrays 
        
        x=elems_dic['s_time']
        y=elems_dic[y_param]
        
        ### Convert raduians to degress
        if y_param=='fast':
            y=y*180/np.pi
            
        ### Change angle mode and lag_model if asked
        
        if y_param=='fast':
            if angle_mode=='azimuth':
                y=swm.trigo2azimuth(y)
        elif y_param=='lag':
            if lag_mode=='ms':
                y=y/sampling_rate*1000
            elif lag_mode=='s':
                y=y/sampling_rate
                
        ### Plot

        (ax,im,X,Y,Z,x_bins,x_diffs)=swm.plot_movehisto2d(x,y,
                x_label=x_label,y_label=y_label,title=title,ax=ax,
                **movehisto2d_kwargs)
        
        ##################################
        ### Plot time windows if asked ###
        ##################################
        
        if window_unit is not None:
            
            ### Modify seconds to proper unit
            
            if window_unit=='minute':
                x_diffs/=60
            elif window_unit=='hour':
                x_diffs/=3600
            elif window_unit=='day':
                x_diffs/=86400
            elif window_unit=='second':
                x_diffs=x_diffs
            else:
                raise ValueError('window unit must be either, day, hour, minute, second')
                
            window_label='Window size [%s]'%window_unit[0:3]
            
            ### Create new ax
            
            ax_win = ax.twinx()
            ax_win.set_yscale("log")
            
            ### Plot 
#            x_start=movehisto2d_kwargs.get('x_start',None)
#            x_end=movehisto2d_kwargs.get('x_end',None)
            
            lw_win=3
            ax_win.plot(x_bins, x_diffs, "k-",lw=lw_win,alpha=0.5)
            ax_win.plot(x_bins, x_diffs, "w-",lw=lw_win/3,alpha=0.5)
#            ax_win.set_xlim(swm.obspytime2matplotlib([x_start,x_end]))
            ### Cosmetic
            
            ax_win.set_ylim(bottom=window_bottom,top=window_top)
            ax_win.set_ylabel(window_label)

        
        ### Add vertical lines associated to start and end of eruption
        
        starteruption_time=UTCDateTime(2015,4,24,6) # Nooner and Chadwick 2016
        enderuption_time=UTCDateTime(2015,5,19)
        vline_times=[starteruption_time,enderuption_time]
        vline_times=swm.obspytime2matplotlib(vline_times)
             
        swm.plot_vlines(vline_times,ax,markercolor='w',markersize=15,
                        markeralpha=0.6,linecolor='w',linewidth=1.5)

        return ax

In [ ]:
def plot_movehisto2d(x,y,
                x_label='X',y_label='Y',title='',ax=None,vmax=None,
                **movehisto2d_kwargs):


    """
    Function made to plot an histo2d but using a moving window in both directionsn, this ensure better
    consistency between neighbor bins. 
    The histogram can also work when data is an obspy.UTCDateTime array, then the x_start and x_end must
    be given in UTCDateTime as well and the x_width should be given in seconds.
    UTCDatetime are converted to timestamps (seconds since 1970)
    
    Inputs
    ------
        x,y: np.array: arrays containing the data to apply histogram on (x can be UTCDateTime)
        x_width,y_width: float: width of the bins (in seconds for UTCDateTime)
        [x,y]_[start,end]: float: start and end for histogram edges
        [x,y]_over: float in [0,1]: overlap for windows [1 = full overlap]
        norm_y: Boolean: True to normalize by the maximum in each column
        smooth: Boolean: True for smoothin (Gaussian Filter)
        gaussian_[x,y]_per: float in [0,100]: width percentage for smoothing (100= filter size equal to data range)
        [x,y]_label: str
        text_list: list,str: titles to be added to the right corner of the figure
        show_counts: bool: If True it will add subplots to shown counts
        
    Ouputs
    ------
        ax_list: plt.axes: object associated to the 4 plots (top,text,mesh,right)
        X,Y,Z: meshes :
        x_bins,x_diffs : x_diffs is in seconds, whereas x_bins in matplotlib time
            x_diffs is the time resolution (i.e. the size of the bins, should be constant
            if windows is used)
    
    Comments:
    ---------
        To be added to the general functions (plot module?)
        x_start and x_end needs to be modified for samples plots
        
    UsedIn
    ------
        SWSCat.plot_movehisto2d_time
    """
    
    ### Check if is made of UTCDateTimes
    
    time_flag=False
    if isinstance(x[0],UTCDateTime):
        time_flag=True
        print('X is in UTCDateTime')
        if movehisto2d_kwargs.get('x_mode','window')=='window':
            print('Remember width should be given in seconds, otherwise memory error')
    
    ### Modify x_start and x_end, and x if x is time and convert to timestamps
    x_start=movehisto2d_kwargs.get('x_start',None)
    x_end=movehisto2d_kwargs.get('x_end',None)
    y_start=movehisto2d_kwargs.get('y_start',None)
    y_end=movehisto2d_kwargs.get('y_end',None)
    
    if time_flag:
        if (x_start is not None) & (not isinstance(x_start,UTCDateTime)):
            raise ValueError('x_start must be given in obspy.UTCDateTime')
        if (x_end is not None) & (not isinstance(x_end,UTCDateTime)):
            raise ValueError('x_end must be given in obspy.UTCDateTime')
        x=np.array([value.timestamp for value in x]) # (seconds since 1970-01-01T00:00:00)
        x_start=x_start.timestamp if x_start is not None else None
        x_end=x_end.timestamp if x_end is not None else None
        movehisto2d_kwargs['x_start']=x_start
        movehisto2d_kwargs['x_end']=x_end

    ### Bin the data
    
    (X,Y,Z,x_bins,x_diffs)=swm.movehisto2d_bin(x,y,**movehisto2d_kwargs)
    
    ########################
    #### Start plotting ####

    #### Grid spec
    
    bottom=0.15 if time_flag else 0.1
    
    ### Checks
    
    if ax is None:
        fig,ax = plt.subplots(gridspec_kw={'bottom':bottom,'left':0.15})
        
    if time_flag:
        X=np.array(swm.timestamp2matplotlib(X.ravel())).reshape(X.shape) # transform for plotting
        x_bins=swm.timestamp2matplotlib(x_bins)
        plt.setp( ax.xaxis.get_majorticklabels(), rotation=30 ,ha='right')
        ax.set_xlim(swm.timestamp2matplotlib([x_start,x_end]))
    else:
        ax.set_xlim([x_start,x_end])
       
    (Xm,Ym)=swm.XY2XY_pcolormesh(X,Y) # To ensure Pcolormesh will be ceneterd on bins

    #im=ax.pcolormesh(X,Y,Z,cmap=plt.cm.get_cmap('jet'),rasterized=True)
    im=ax.pcolormesh(Xm,Ym,Z,cmap=plt.cm.get_cmap('magma'),rasterized=True,vmax=vmax)
    
    ax.set_ylim([y_start,y_end])
    
    ax.set_aspect('auto')
    if time_flag:
        ax.xaxis_date()
            
    ### Cosmetic
    
    ax.set_ylabel(y_label) 
    if not time_flag:
        ax.set_xlabel(x_label) 
   
    ###### Return
    
    return (ax,im,X,Y,Z,x_bins,x_diffs)

In [ ]:
# Plot moving histogram 2D with origin_time vs dt
import sys
sys.path.insert(0, '/Users/mhemmett/Seismology/axial-splitting-ml/scripts')
import sws_methods as swm

# Extract origin_time and dt from results_swspy
origin_times = []
dts = []

for eid, result_data in results_swspy.items():
    result = result_data['result']
    
    # Get origin_time (convert to UTCDateTime if needed)
    origin_time = result.get('origin_time')
    if origin_time is not None:
        if not isinstance(origin_time, UTCDateTime):
            origin_time = UTCDateTime(origin_time)
        
        # Get dt (delay time)
        dt = result.get('dt')
        
        # Only include if both values are valid
        if dt is not None and not np.isnan(dt):
            origin_times.append(origin_time)
            dts.append(dt)

# Convert to numpy arrays
origin_times = np.array(origin_times)
dts = np.array(dts) 

print(f"Number of valid measurements: {len(origin_times)}")
print(f"Time range: {origin_times.min()} to {origin_times.max()}")
print(f"dt range: {dts.min():.3f}s to {dts.max():.3f}s")

# Create the plot
fig, ax = plt.subplots(figsize=(12, 6))

# Call plot_movehisto2d with appropriate parameters
(ax, im, X, Y, Z, x_bins, x_diffs) = swm.plot_movehisto2d(
    origin_times, 
    dts,
    x_label='Time',
    y_label='Delay Time δt (s)',
    title='Shear-Wave Splitting: δt vs Time',
    ax=ax,
    x_mode='window',
    x_width=86400*30,  # 30 days in seconds
    x_over=0.5,  # 50% overlap
    y_mode='bin',
    y_width=0.01,  # 0.01s bins
    y_start=0,
    y_end=0.2,  # Adjust based on your data range
    norm_y=True,
    smooth=True,
    gaussian_x_per=5,
    gaussian_y_per=5
)

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Normalized Count')

plt.tight_layout()
plt.show()